# Java vs Python — Evaluation Engine Comparison

Runs both engines against an OpenML server and compares their outputs. Point it at **test.openml.org** or your local `services` docker-compose stack (see `STACK` below).
**Java is the original/source of truth; the Python port is configured to match it.**

Prerequisites:
- Python side: the `EvaluationEnginePython` project (invoked via `uv` by the cells below).
- Java side: build the jar once — `cd EvaluationEngine && mvn clean package -DskipTests` — then set `JAVA_JAR`.
- A **test.openml.org API key** (`TEST_API_KEY`) is required for the *Java* upload paths (`evaluate_run`, `process_dataset`) when `STACK="test"`. Read-only paths work without one.
- **Or run against the local stack** (the `services` repo's docker-compose): set `STACK="local"` and bring it up with `cd <services> && docker compose --profile evaluation-engine up -d`. It uses the fixed api key `abc` and serves the REST API at `http://localhost:8080/`.

Two conveniences over a plain side-by-side run:

- **Python never uploads.** `run_py` passes the `-no-upload` flag (flip `PY_NO_UPLOAD` to disable). Under it, `evaluate_run` computes the scores locally and prints the evaluation XML to stdout instead of POSTing — so the test server is only mutated by Java, and the two engines no longer clobber each other's results on the same run.
- **Multiple groups at once.** `GROUPS` is a list of `{task, run, dataset, merge}` dicts; every comparison cell below loops over all of them. Add as many rows as you like (leave `merge` empty to skip the merge test for that row).

The notebook kernel only needs the Python standard library (it shells out to `uv` / `java`); the read-back helpers use `urllib` + `xml.etree`, no extra packages.

In [1]:
# ---- EDIT THESE -----------------------------------------------------------
JAVA_JAR = "/Users/smukherjee/Documents/Projects/OpenML/EvaluationEngine/target/EvaluationEngine-1.2.2-SNAPSHOT-jar-with-dependencies.jar"
PY_PROJECT = "/Users/smukherjee/Documents/Projects/OpenML/EvaluationEnginePython"
# Which OpenML instance both engines talk to (and the read-back helpers hit):
#   "test"  -> public test server https://test.openml.org/ (needs TEST_API_KEY below)
#   "local" -> the services docker-compose stack on your machine (fixed api key "abc").
#              Bring it up first:  cd <services> && docker compose --profile evaluation-engine up -d
STACK = "local"           # "test" or "local"
TEST_SERVER = {
    "test":  "https://test.openml.org/",
    "local": "http://localhost:8080/",     # local php-api (host port 8080)
}[STACK]
TEST_API_KEY = ""        # your test.openml.org key (JAVA uploads). Auto-set to "abc" for STACK="local".
if STACK == "local":
    TEST_API_KEY = "abc"

# Python never uploads to the test server — run_py passes the -no-upload flag.
# Set this False if you ever want Python to upload.
PY_NO_UPLOAD = True

DATA_FORMAT = "arff"      # Python dataset format: "arff" (default) or "parquet" — passed via `-df`

# Groups of ids to compare. Each row runs all four functions against its ids.
# Add as many rows as you like; leave "merge" empty ("") to skip merge_datasets
# for that row. The defaults below are common ids on test.openml.org.
GROUPS = [
    {"task": 236, "run": 6, "dataset": 40, "merge": ""},   # task 236 = 10-fold CV; dataset 40
    # {"task": 1, "run": 1, "dataset": 1, "merge": ""},      # add more groups here
]
# --------------------------------------------------------------------------
assert GROUPS, "add at least one group to GROUPS"

## Helpers
Run-once utility cells: shell out to each engine and compare outputs.

In [2]:
import subprocess, pathlib, difflib
import urllib.request, xml.etree.ElementTree as ET

NS = {"oml": "http://openml.org/openml"}
_QUOTES = "'" + '"'        # both quote chars, for stripping nominal values

def _cfg(api_key=None):
    return f"api_key={api_key if api_key is not None else TEST_API_KEY};server={TEST_SERVER}"

def run_java(fn, *opts, api_key=None):
    jar = pathlib.Path(JAVA_JAR)
    if not jar.exists():
        print(f"!! jar not found: {JAVA_JAR}")
        print("   build it: cd <EvaluationEngine> && mvn clean package -DskipTests")
        return None
    cmd = ["java", "-jar", str(jar), "-config", _cfg(api_key), "-f", fn, *map(str, opts)]
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout.strip(): print("[java stdout]", r.stdout.strip()[-1200:])
    if r.stderr.strip(): print("[java stderr]", r.stderr.strip()[-1200:])
    return r

def run_py(fn, *opts, test=True, no_upload=PY_NO_UPLOAD):
    """Shell out to the Python port. When ``no_upload`` is True (default,
    controlled by PY_NO_UPLOAD) the -no-upload flag is passed so evaluate_run
    computes scores locally and prints the evaluation XML to stdout instead
    of uploading."""
    cmd = ["uv", "run", "--project", PY_PROJECT, "python", "-m", "main",
           "-f", fn, "-df", DATA_FORMAT, *(["-test"] if test else []),
           *(["-no-upload"] if no_upload else []),
           *map(str, opts)]
    print("$", " ".join(cmd),
          "  [-no-upload]" if no_upload else "")
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=PY_PROJECT)
    if r.stdout.strip(): print("[py stdout]", r.stdout.strip()[-1200:])
    if r.stderr.strip(): print("[py stderr]", r.stderr.strip()[-1200:])
    return r

# ---- splits ARFF comparison (robust to column order / quoting) ------------
def parse_splits(path):
    attrs, rows, in_data = [], set(), False
    for line in pathlib.Path(path).read_text().splitlines():
        s, up = line.strip(), line.strip().upper()
        if up.startswith("@ATTRIBUTE"):
            attrs.append(s.split(None, 2)[1].strip(_QUOTES).lower())
        elif up == "@DATA":
            in_data = True
        elif in_data and s and not s.startswith("%"):
            vals = [x.strip().strip(_QUOTES) for x in s.split(",")]
            row = dict(zip(attrs, vals))
            rows.add((row.get("type"), row.get("rowid"), row.get("repeat"),
                      row.get("fold"), row.get("sample")))
    return rows

def compare_splits(java_path, py_path):
    if not (pathlib.Path(java_path).exists() and pathlib.Path(py_path).exists()):
        print("missing output file(s)"); return
    j, p = parse_splits(java_path), parse_splits(py_path)
    oj, op = j - p, p - j
    print(f"java rows={len(j)}  py rows={len(p)}  only_java={len(oj)}  only_py={len(op)}")
    if oj: print("sample java-only:", sorted(oj)[:5])
    if op: print("sample py-only:", sorted(op)[:5])
    print("=> MATCH" if not oj and not op else "=> MISMATCH")

def diff_files(a, b, la="java", lb="python", n=40):
    if not (pathlib.Path(a).exists() and pathlib.Path(b).exists()):
        print("missing output file(s)"); return
    xa = pathlib.Path(a).read_text().splitlines()
    xb = pathlib.Path(b).read_text().splitlines()
    d = list(difflib.unified_diff(xa, xb, fromfile=la, tofile=lb, n=1))
    print(f"{la}: {len(xa)} lines   {lb}: {len(xb)} lines")
    print("IDENTICAL" if not d else f"DIFFER ({len(d)} diff lines), first {n}:")
    for line in d[:n]: print(line)

def arff_data_rows(path):
    rows, in_data = [], False
    for line in pathlib.Path(path).read_text().splitlines():
        s = line.strip()
        if s.upper() == "@DATA": in_data = True
        elif in_data and s and not s.startswith("%"):
            rows.append(tuple(x.strip().strip(_QUOTES) for x in s.split(",")))
    return rows

# ---- evaluation comparison ----------------------------------------------
# Server read-back (Java uploads, we fetch what it stored).
def _get_xml(url):
    with urllib.request.urlopen(url) as r:
        return ET.fromstring(r.read())

def _parse_array(s):
    if not s:
        return None
    return [float(x) for x in s.strip("[]").split(",") if x.strip() != ""]

def fetch_evals(run_id):
    """{function: {"value": ..., "array_data": ...}} as stored on test.

    The /evaluation/list/run endpoint returns one row per metric (the global
    aggregate), with per-fold values in ``array_data``."""
    root = _get_xml(f"{TEST_SERVER}api/v1/xml/evaluation/list/run/{run_id}")
    out = {}
    for e in root.findall("oml:evaluation", NS):
        out[e.findtext("oml:function", "", NS)] = {
            "value": e.findtext("oml:value", None, NS),
            "array_data": e.findtext("oml:array_data", None, NS),
        }
    return out

def run_eval_error(xml_str):
    """Return the <oml:error> text from a no-upload evaluate_run XML, or None."""
    try:
        root = ET.fromstring(xml_str.strip().encode())
    except ET.ParseError:
        return None
    return root.findtext("oml:error", None, NS)

def parse_run_eval_xml(xml_str):
    """Parse the <oml:run_evaluation> XML printed by ``evaluate_run`` with
    -no-upload into the same {function: {"value", "array_data"}} shape
    that ``fetch_evals`` returns from the server, so ``evals_diff`` works for
    both. Global scores (no repeat/fold/sample) give the value; per-cell scores
    are ordered by (repeat, fold, sample) and joined into array_data."""
    root = ET.fromstring(xml_str.strip().encode())
    per_cell, globals_, arrays = {}, {}, {}
    for e in root.findall("oml:evaluation", NS):
        name = e.findtext("oml:name", "", NS)
        value = e.findtext("oml:value", None, NS)
        repeat = e.attrib.get("repeat")
        fold = e.attrib.get("fold")
        sample = e.attrib.get("sample")
        if repeat is None and fold is None and sample is None:
            globals_[name] = value
            ad = e.findtext("oml:array_data", None, NS)
            if ad is not None:
                arrays[name] = ad
        else:
            key = (int(repeat or 0), int(fold or 0), int(sample or 0))
            per_cell.setdefault(name, []).append((key, value))
    out = {}
    for name in set(globals_) | set(per_cell):
        ad = arrays.get(name)
        if ad is None and name in per_cell:
            vals = [v for _, v in sorted(per_cell[name]) if v is not None]
            ad = ",".join(vals) if vals else None
        out[name] = {"value": globals_.get(name), "array_data": ad}
    return out

def evals_diff(j, p, label=""):
    jn, pn = set(j), set(p)
    print(f"=== {label}: {len(jn)} java metrics, {len(pn)} py metrics; names match: {jn == pn} ===")
    if jn - pn: print("  only java:", sorted(jn - pn))
    if pn - jn: print("  only py :", sorted(pn - jn))
    diffs = []
    for name in sorted(jn & pn):
        jv, pv = j[name]["value"], p[name]["value"]
        ja, pa = _parse_array(j[name]["array_data"]), _parse_array(p[name]["array_data"])
        differ = False
        try:
            if abs(float(jv) - float(pv)) > 1e-9:
                differ = True
        except (TypeError, ValueError):
            if jv != pv:
                differ = True
        if ja is not None or pa is not None:
            if ja != pa and (ja is None or pa is None or len(ja) != len(pa)
                             or any(abs(a - b) > 1e-9 for a, b in zip(ja, pa))):
                differ = True
        if differ:
            diffs.append(name)
            print(f"  {name}: value java={jv} py={pv} | array java={j[name]['array_data']} py={p[name]['array_data']}")
    print("=> all metrics MATCH" if not diffs and jn == pn else f"=> {len(diffs)} metric(s) differ")

def fetch_features(did):
    return _features_from(_get_xml(f"{TEST_SERVER}api/v1/xml/data/features/{did}"))

def parse_features_xml(xml_str):
    return _features_from(ET.fromstring(xml_str.strip().encode()))

def _features_from(root):
    out = {}
    for f in root.findall("oml:feature", NS):
        name = f.findtext("oml:name", "", NS)
        fields, nominals = {}, []
        for c in f:
            local = c.tag.split("}", 1)[-1].removeprefix("oml:")
            if local == "nominal_value":
                nominals.append(c.text)
            else:
                fields[local] = c.text
        fields["nominal_value"] = nominals
        out[name] = fields
    return out

def features_diff(j, p):
    jn, pn = set(j), set(p)
    print(f"feature names match: {jn == pn}  (only_java={len(jn-pn)}, only_py={len(pn-jn)})")
    if jn - pn: print("  only java:", sorted(jn - pn))
    if pn - jn: print("  only py :", sorted(pn - jn))
    ndiff = 0
    for name in sorted(jn & pn):
        for key in sorted(set(j[name]) | set(p[name])):
            jv, pv = j[name].get(key), p[name].get(key)
            if key == "nominal_value":
                jv, pv = sorted(x for x in (jv or []) if x), sorted(x for x in (pv or []) if x)
            if jv != pv:
                ndiff += 1; print(f"  {name}.{key}: java={jv!r}  py={pv!r}")
    print("=> features MATCH" if ndiff == 0 and jn == pn else f"=> {ndiff} feature field(s) differ")

print("helpers loaded")

helpers loaded


## 1. generate_folds  *(task id → splits ARFF)*
Both take `-id <task_id>`; folds/repeats/type come from the task's estimation procedure, seed 0 (Java `Main.FOLD_GENERATION_SEED`). Loops over every `task` id in `GROUPS`.

In [5]:
# Java (source of truth) — one run per group.
for g in GROUPS:
    run_java("generate_folds", "-id", g["task"], "-o", f"/tmp/java_folds_{g['task']}.arff")

$ java -jar /Users/smukherjee/Documents/Projects/OpenML/EvaluationEngine/target/EvaluationEngine-1.2.2-SNAPSHOT-jar-with-dependencies.jar -config api_key=abc;server=http://localhost:8080/ -f generate_folds -id 236 -o /tmp/java_folds_236.arff
[java stderr] WARNING: A terminally deprecated method in sun.misc.Unsafe has been called


In [6]:
# Python (configured to match Java: task id, task-derived procedure, seed 0).
for g in GROUPS:
    run_py("generate_folds", "-id", g["task"], "-o", f"/tmp/py_folds_{g['task']}.arff")

$ uv run --project /Users/smukherjee/Documents/Projects/OpenML/EvaluationEnginePython python -m main -f generate_folds -df arff -test -no-upload -id 236 -o /tmp/py_folds_236.arff   [-no-upload]
[py stdout] wrote 5000 split rows to /tmp/py_folds_236.arff


In [7]:
# Compare split assignments (TRAIN/TEST per rowid x repeat x fold),
# ignoring ARFF formatting differences.
for g in GROUPS:
    print(f"--- task {g['task']} ---")
    compare_splits(f"/tmp/java_folds_{g['task']}.arff", f"/tmp/py_folds_{g['task']}.arff")

--- task 236 ---
java rows=5000  py rows=5000  only_java=2624  only_py=2624
sample java-only: [('TEST', '0', '0', '0', None), ('TEST', '0', '2', '0', None), ('TEST', '0', '3', '0', None), ('TEST', '0', '4', '0', None), ('TEST', '1', '2', '1', None)]
sample py-only: [('TEST', '0', '0', '1', None), ('TEST', '0', '2', '1', None), ('TEST', '0', '3', '1', None), ('TEST', '0', '4', '1', None), ('TEST', '1', '2', '0', None)]
=> MISMATCH


## 2. evaluate_run  *(run id → metric scores)*
Java computes metrics for the run's predictions and **uploads** to test; we read the stored evaluations back. Python computes the **same metrics locally and does NOT upload** (`-no-upload` via `run_py`); its evaluation XML is parsed straight from stdout. We then compare per metric, looping over every `run` id in `GROUPS`.

**Note:** `kb_relative_information_score` is flagged in the port as possibly diverging from Weka.

In [13]:
# Java evaluates + uploads. (Needs TEST_API_KEY with write access.)
JAVA_EVALS = {}
for g in GROUPS:
    run_java("evaluate_run", "-id", g["run"])
    try:
        JAVA_EVALS[g["run"]] = fetch_evals(g["run"])
    except Exception as e:
        print(f"  !! could not read back run {g['run']}: {e}")
        JAVA_EVALS[g["run"]] = {}
print("java metric sets:", {r: len(v) for r, v in JAVA_EVALS.items()})

$ java -jar /Users/smukherjee/Documents/Projects/OpenML/EvaluationEngine/target/EvaluationEngine-1.2.2-SNAPSHOT-jar-with-dependencies.jar -config api_key=abc;server=http://localhost:8080/ -f evaluate_run -id 6
[java stderr]  terminally deprecated method in sun.misc.Unsafe has been called
[04-08-2026 15:40:17] [OK] [Process Run] Start processing run: 6
org.openml.apiconnector.io.ApiException: Run not found
	at org.openml.apiconnector.io.HttpConnector.wrapHttpResponse(HttpConnector.java:192)
	at org.openml.apiconnector.io.HttpConnector.doApiGetRequest(HttpConnector.java:106)
	at org.openml.apiconnector.io.OpenmlBasicConnector.runGet(OpenmlBasicConnector.java:922)
	at org.openml.webapplication.EvaluateRun.evaluate(EvaluateRun.java:100)
	at org.openml.webapplication.EvaluateRun.<init>(EvaluateRun.java:55)
	at org.openml.webapplication.Main.main(Main.java:112)
[04-08-2026 15:40:17] [Error] [Main] {"error":"Run not found"}
  !! could not read back run 6: HTTP Error 412: Precondition Failed
j

In [14]:
# Python evaluates locally (no upload — -no-upload passed by run_py).
PY_EVALS = {}
for g in GROUPS:
    rpy = run_py("evaluate_run", "-id", g["run"])
    if not rpy or rpy.returncode != 0:
        PY_EVALS[g["run"]] = {}
        continue
    err = run_eval_error(rpy.stdout)
    if err:
        print(f"  !! run {g['run']} computed with error: {err}")
    PY_EVALS[g["run"]] = parse_run_eval_xml(rpy.stdout)
print("py metric sets:", {r: len(v) for r, v in PY_EVALS.items()})

$ uv run --project /Users/smukherjee/Documents/Projects/OpenML/EvaluationEnginePython python -m main -f evaluate_run -df arff -test -no-upload -id 6   [-no-upload]
[py stdout] <?xml version="1.0" encoding="utf-8"?>
<oml:run_evaluation xmlns:oml="http://openml.org/openml"><oml:run_id>6</oml:run_id><oml:evaluation_engine_id>1</oml:evaluation_engine_id><oml:error>404 Client Error: Not Found for url: https://test.openml.org/data/download/159/Run_6_predictions.arff</oml:error></oml:run_evaluation>
  !! run 6 computed with error: 404 Client Error: Not Found for url: https://test.openml.org/data/download/159/Run_6_predictions.arff
py metric sets: {6: 0}


In [ ]:
for g in GROUPS:
    rid = g["run"]
    evals_diff(JAVA_EVALS.get(rid, {}), PY_EVALS.get(rid, {}),
               label=f"evaluate_run run={rid}")

## 3. process_dataset  *(dataset id → feature XML)*
Java `process_dataset` uploads features to test; Python has a read-only `process_dataset_print` that prints the same XML (and never uploads). We compare Python's printed features against the features Java stored on test, looping over every `dataset` id in `GROUPS`.

In [8]:
# Python: extract features locally, print XML (no upload, no key needed).
PY_FEAT = {}
for g in GROUPS:
    rpy = run_py("process_dataset_print", "-id", g["dataset"])
    PY_FEAT[g["dataset"]] = parse_features_xml(rpy.stdout) if rpy else {}
print("py feature sets:", {d: len(v) for d, v in PY_FEAT.items()})

$ uv run --project /Users/smukherjee/Documents/Projects/OpenML/EvaluationEnginePython python -m main -f process_dataset_print -df arff -test -no-upload -id 40   [-no-upload]
[py stdout] real_values>0</oml:number_of_real_values>
		<oml:number_of_values>500</oml:number_of_values>
		<oml:maximum_value>75.0</oml:maximum_value>
		<oml:minimum_value>18.0</oml:minimum_value>
		<oml:mean_value>38.934599156118146</oml:mean_value>
		<oml:standard_deviation>15.317523442690833</oml:standard_deviation>
	</oml:feature>
	<oml:feature>
		<oml:index>5</oml:index>
		<oml:name>Type_school</oml:name>
		<oml:data_type>nominal</oml:data_type>
		<oml:nominal_value>primary_terminal_leaver</oml:nominal_value>
		<oml:nominal_value>secondary</oml:nominal_value>
		<oml:nominal_value>vocational</oml:nominal_value>
		<oml:is_target>false</oml:is_target>
		<oml:is_ignore>false</oml:is_ignore>
		<oml:is_row_identifier>false</oml:is_row_identifier>
		<oml:number_of_missing_values>0</oml:number_of_missing_values>
		<om

In [9]:
# Java: extract + upload features to test. (Needs TEST_API_KEY.)
JAVA_FEAT = {}
for g in GROUPS:
    run_java("process_dataset", "-id", g["dataset"])
    try:
        JAVA_FEAT[g["dataset"]] = fetch_features(g["dataset"])
    except Exception as e:
        print(f"  !! could not read back dataset {g['dataset']}: {e}")
        JAVA_FEAT[g["dataset"]] = {}
print("java feature sets:", {d: len(v) for d, v in JAVA_FEAT.items()})

$ java -jar /Users/smukherjee/Documents/Projects/OpenML/EvaluationEngine/target/EvaluationEngine-1.2.2-SNAPSHOT-jar-with-dependencies.jar -config api_key=abc;server=http://localhost:8080/ -f process_dataset -id 40
[java stderr] works.xstream.converters.reflection.SunUnsafeReflectionProvider
[04-08-2026 15:34:08] [OK] [Process Dataset] Processing dataset 40 on special request. 
[04-08-2026 15:34:08] [OK] [URL] HTTP request status code [200] URL [http://localhost:8080/data/v1/download/40/irish.arff?api_key=abc]
[04-08-2026 15:34:08] [OK] [Process Dataset] Processing dataset 40 - obtaining features. 
[04-08-2026 15:34:08] [OK] [Process Dataset] Processing dataset 40 - obtaining basic qualities. 
[04-08-2026 15:34:08] [OK] [Download] Start downloading dataset: 40
[04-08-2026 15:34:08] [OK] [Extract Features] Start extracting features for dataset: 40
[04-08-2026 15:34:08] [OK] [Extract Features] Running Batch Characterizers (full data, might take a while)
[04-08-2026 15:34:08] [OK] [Extract

In [12]:
PY_FEAT

{40: {'Sex': {'index': '0',
   'name': 'Sex',
   'data_type': 'nominal',
   'is_target': 'false',
   'is_ignore': 'false',
   'is_row_identifier': 'false',
   'number_of_missing_values': '0',
   'number_of_distinct_values': '2',
   'number_of_unique_values': '0',
   'number_of_nominal_values': '500',
   'number_of_values': '500',
   'class_distribution': 'female:250,male:250',
   'nominal_value': ['female', 'male']},
  'DVRT': {'index': '1',
   'name': 'DVRT',
   'data_type': 'numeric',
   'is_target': 'false',
   'is_ignore': 'false',
   'is_row_identifier': 'false',
   'number_of_missing_values': '0',
   'number_of_distinct_values': '68',
   'number_of_unique_values': '10',
   'number_of_integer_values': '500',
   'number_of_real_values': '0',
   'number_of_values': '500',
   'maximum_value': '140.0',
   'minimum_value': '65.0',
   'mean_value': '100.152',
   'standard_deviation': '15.440883912522624',
   'nominal_value': []},
  'Educational_level': {'index': '2',
   'name': 'Educati

In [11]:
JAVA_FEAT

{40: {'Sex': {'index': '0',
   'name': 'Sex',
   'data_type': 'nominal',
   'is_target': 'false',
   'is_ignore': 'false',
   'is_row_identifier': 'false',
   'number_of_missing_values': '0',
   'nominal_value': ['female', 'male']},
  'DVRT': {'index': '1',
   'name': 'DVRT',
   'data_type': 'numeric',
   'is_target': 'false',
   'is_ignore': 'false',
   'is_row_identifier': 'false',
   'number_of_missing_values': '0',
   'nominal_value': []},
  'Educational_level': {'index': '2',
   'name': 'Educational_level',
   'data_type': 'nominal',
   'is_target': 'false',
   'is_ignore': 'false',
   'is_row_identifier': 'false',
   'number_of_missing_values': '6',
   'nominal_value': ['3rd_level_complete',
    '3rd_level_incomplete',
    'Junior_cycle_incomplete-secondary_school',
    'Junior_cycle_incomplete-vocational_school',
    'Junior_cycle_terminal_leaver-secondary_school',
    'Junior_cycle_terminal_leaver-vocational_school',
    'Primary_terminal_leaver',
    'Senior_cycle_incomplete-s

In [10]:
for g in GROUPS:
    did = g["dataset"]
    print(f"--- dataset {did} ---")
    features_diff(JAVA_FEAT.get(did, {}), PY_FEAT.get(did, {}))

--- dataset 40 ---
feature names match: True  (only_java=0, only_py=0)
  DVRT.maximum_value: java=None  py='140.0'
  DVRT.mean_value: java=None  py='100.152'
  DVRT.minimum_value: java=None  py='65.0'
  DVRT.number_of_distinct_values: java=None  py='68'
  DVRT.number_of_integer_values: java=None  py='500'
  DVRT.number_of_real_values: java=None  py='0'
  DVRT.number_of_unique_values: java=None  py='10'
  DVRT.number_of_values: java=None  py='500'
  DVRT.standard_deviation: java=None  py='15.440883912522624'
  Educational_level.class_distribution: java=None  py='3rd_level_complete:57,3rd_level_incomplete:7,Junior_cycle_incomplete-secondary_school:30,Junior_cycle_incomplete-vocational_school:50,Junior_cycle_terminal_leaver-secondary_school:65,Junior_cycle_terminal_leaver-vocational_school:68,Primary_terminal_leaver:37,Senior_cycle_incomplete-secondary_school:9,Senior_cycle_incomplete-vocational_school:13,Senior_cycle_terminal_leaver-secondary_school:158'
  Educational_level.number_of_dis

## 4. merge_datasets  *(MultiTask task id → merged ARFF)*
Uses the `merge` id from each group (rows with an empty `merge` are skipped). Both write the merged ARFF to a file.

In [15]:
MERGE_GROUPS = [g for g in GROUPS if g["merge"]]
assert MERGE_GROUPS, "set 'merge' to a real MultiTask task id in at least one GROUP"
for g in MERGE_GROUPS:
    run_java("merge_datasets", "-id", g["merge"], "-o", f"/tmp/java_merged_{g['merge']}.arff")

AssertionError: set 'merge' to a real MultiTask task id in at least one GROUP

In [16]:
for g in MERGE_GROUPS:
    run_py("merge_datasets", "-id", g["merge"], "-o", f"/tmp/py_merged_{g['merge']}.arff")

In [ ]:
# Merged ARFFs differ cosmetically (Weka vs liac-arff); compare row sets.
for g in MERGE_GROUPS:
    mid = g["merge"]
    print(f"--- merge task {mid} ---")
    diff_files(f"/tmp/java_merged_{mid}.arff", f"/tmp/py_merged_{mid}.arff")
    jr = set(arff_data_rows(f"/tmp/java_merged_{mid}.arff"))
    pr = set(arff_data_rows(f"/tmp/py_merged_{mid}.arff"))
    print(f"java data rows={len(jr)}  py data rows={len(pr)}  only_java={len(jr-pr)} only_py={len(pr-jr)}")
    print("=> row sets MATCH" if jr == pr else "=> row sets DIFFER")

## 5. Not yet ported to Python

`all_wrong`, `different_predictions` (Java: `InstanceBased.java`) and `challenge` (Java: `ChallengeSets.java`) are **NotImplemented** in the Python port — each pair below runs the Java command and confirms the Python side raises `NotImplementedError`. They loop over the `task` (and `run`) ids in `GROUPS`.


### all_wrong  *(needs `-r` run ids + `-t` task id)*

In [ ]:
# Java
for g in GROUPS:
    run_java("all_wrong", "-r", g["run"], "-t", g["task"])

In [ ]:
# Python — NotImplemented
for g in GROUPS:
    run_py("all_wrong", "-r", g["run"], "-t", g["task"])

### different_predictions  *(needs `-r` run ids + `-t` task id)*

In [ ]:
# Java
for g in GROUPS:
    run_java("different_predictions", "-r", g["run"], "-t", g["task"])

In [ ]:
# Python — NotImplemented
for g in GROUPS:
    run_py("different_predictions", "-r", g["run"], "-t", g["task"])

### challenge  *(needs `-t` task id + `-mode train|test`)*

In [ ]:
# Java
for g in GROUPS:
    run_java("challenge", "-t", g["task"], "-mode", "train")

In [ ]:
# Python — NotImplemented
for g in GROUPS:
    run_py("challenge", "-t", g["task"], "-mode", "train")